# YOLOv5 data validation on SoccerTrack v2 (match 117093)

Pre-SAM sanity check: convert GSR ground-truth boxes → a YOLOv5 dataset and train
YOLOv5 on Colab's GPU to confirm the data is sound and a detector can learn players.

**Logic lives in `src/`** (repo rule — CLAUDE.md §2). This notebook only orchestrates.

Order: setup → download ONE video → **verify box alignment** → build dataset → train → predict.

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. Setup — clone repo + install deps

In [ ]:
import os, subprocess
REPO = 'https://github.com/wheredawoodat949/AI-Hackathon.git'
BRANCH = 'feat/pitch-eval-shaaz'  # branch carrying the YOLO converter
if not os.path.isdir('AI-Hackathon'):
    subprocess.run(['git','clone','--branch',BRANCH,REPO], check=True)
else:
    subprocess.run(['git','-C','AI-Hackathon','pull'], check=True)
%cd AI-Hackathon
!nvidia-smi -L

In [ ]:
# Data deps for download + converter (no torch needed yet; ultralytics pulls its own).
!pip -q install gdown PyYAML python-dotenv numpy opencv-python-headless pillow
# Make `import src...` work from the repo root.
import sys; sys.path.insert(0, os.getcwd())

## 2. Download ONE video (normal 2nd-half panorama only)

The 1st-half video in the mirror is a 47-frame stub; the **2nd half is complete**
(~70k frames). `--video-pattern panorama_2nd` fetches only that ~3GB file (plus the
small gsr/bas/raw), skipping the other ~5GB of video. `panorama_2nd` matches the
**normal** panorama, not `calibrated_panorama` (whose pixels don't match the boxes).

In [ ]:
!python -m src.data.download --match 117093 --source drive --video-pattern panorama_2nd
!ls -lh data/videos/117093/ data/gsr/117093/

## 3. ⚠️ VERIFY box alignment BEFORE exporting

GSR `bbox_image` boxes did **not** align with the *calibrated* panorama. This draws
boxes on real frames of the **normal** panorama. **Look at the images below: do the
green boxes sit on players?**
- ✅ Yes → the video's native size is the right normalization basis; go to step 4.
- ❌ No  → boxes are in a different image space. Try the remap in step 3b before exporting.

In [ ]:
import glob
from src.data.yolo_dataset import overlay_check
VIDEO = sorted(glob.glob('data/videos/117093/*panorama_2nd*'))[0]
print('video:', VIDEO)
pngs = overlay_check('data', '117093', VIDEO, half=2, image_ids=(1000, 20000, 50000))

from IPython.display import Image, display
for p in pngs:
    print(p); display(Image(filename=str(p)))

### 3b. (only if boxes were misaligned) remap frames to GSR space

`raw/117093_mapx.npy` + `mapy.npy` un-warp the panorama into the GSR's native image
space, so the raw boxes line up. Run this, re-check, and if aligned pass the remapped
size to `build(..., img_wh=...)`. Skip this cell if step 3 already looked correct.

In [ ]:
import numpy as np, cv2
from src.data.loader import load_match
mapx = np.load('data/raw/117093_mapx.npy'); mapy = np.load('data/raw/117093_mapy.npy')
print('remap target size (WxH):', mapx.shape[1], 'x', mapx.shape[0])
cap = cv2.VideoCapture(VIDEO); cap.set(cv2.CAP_PROP_POS_FRAMES, 20000-1)
ok, img = cap.read(); cap.release()
remapped = cv2.remap(img, mapx, mapy, cv2.INTER_LINEAR)
fr = load_match('data','117093').gsr_frame(2, 20000)
for e in fr.entities:
    if e.bbox_image:
        x,y,w,h = e.bbox_image; cv2.rectangle(remapped,(x,y),(x+w,y+h),(80,200,80),2)
cv2.imwrite('outputs/remap_check.png', remapped)
from IPython.display import Image, display; display(Image('outputs/remap_check.png'))

## 4. Build the YOLOv5 dataset

`stride=25` ≈ 1 fps; `limit` caps frames for a quick run. Bump both for a fuller set.
If step 3 needed the remap, set `img_wh=(mapx.shape[1], mapx.shape[0])`.

In [ ]:
from src.data.yolo_dataset import build
build('data', '117093', VIDEO, out_dir='yolo_ds',
      half=2, stride=25, limit=400, img_wh=None)  # img_wh=None -> video native size
!cat yolo_ds/data.yaml && echo '---' && ls yolo_ds/images/train | head

## 5. Train YOLOv5 (Ultralytics)

Ultralytics serves YOLOv5 weights (`yolov5su.pt`) through the same API. A few epochs
is enough to prove the data trains — this is a validation run, not a final model.

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO
model = YOLO('yolov5su.pt')  # YOLOv5-small (Ultralytics build)
model.train(data='yolo_ds/data.yaml', epochs=15, imgsz=1280, batch=8,
            project='runs', name='st_yolov5', exist_ok=True)

## 6. Validate — predict on a held-out frame

Run the trained model on a val image and view the detections. Boxes on players =
the data validates end-to-end and a detector learns it. ✅

In [ ]:
import glob
val_imgs = sorted(glob.glob('yolo_ds/images/val/*.jpg'))
res = model.predict(val_imgs[0], imgsz=1280, conf=0.25, save=True, project='runs', name='pred', exist_ok=True)
print('detections:', len(res[0].boxes))
from IPython.display import Image, display
display(Image(filename=sorted(glob.glob('runs/pred/*.jpg'))[0]))

## Next
- If detections look right, the GSR data + bbox pipeline are validated for SAM/Phase 2.
- `metrics = model.val()` gives mAP for an Arize before/after story (see docs/ML_DIRECTIONS.md).
- The same boxes feed Role B's homography → minimap (raw/117093_homography.npy).